# Reinforcement Learning Fine-Tuning (RLFT) on GEAP

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package rather than re-implementing logic. See
[`docs/notes/tuning-apis.md`](../docs/notes/tuning-apis.md) for the API details.

Where [SFT](01_sft.ipynb) taught the model *what* to answer and
[DPO](02_preference_tuning.ipynb) taught it *how* to phrase a reply, **RLFT**
rewards *correctness*. Each record carries a `references` dict of ground-truth
metadata and **no** gold completion; the model draws `samples_per_prompt`
candidate generations that a **reward function** scores. It uses the *same*
`client.tunings.tune(...)` call, with `method="REINFORCEMENT_TUNING"` and a
`reward_config`.

> **Best practice:** SFT first, then continuous-tune from that checkpoint with
> RLFT. This demo tunes the base model directly to stay self-contained.

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place before running the tune/eval cells.

In [1]:
from geap_tuning.config import genai_client, load_config

# We tune gemini-3.5-flash, but the client stays REGIONAL. Gemini 3.x is served
# from the `global` endpoint for *inference*, but `global` does not support
# tuning (or validate_reward) — those run in a region (us-central1/europe-west4).
BASE_MODEL = "gemini-3.5-flash"
cfg = load_config()
client = genai_client(cfg)
cfg

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


TuningConfig(project='hybrid-vertex', location='us-central1', bucket='gs://geap-tuning-hybrid-vertex')

## 1. Build the verifiable-math dataset

Deterministic train/val/test splits of `(question, ground_truth_answer)` pairs.
Each record ends on a **user** turn and carries the answer in `references` — no
gold model turn.

In [2]:
from geap_tuning.rlft.data import build_rlft_dataset

paths = build_rlft_dataset("../datasets/rlft_math")
paths

{'train': '../datasets/rlft_math/train.jsonl',
 'val': '../datasets/rlft_math/val.jsonl',
 'test': '../datasets/rlft_math/test.jsonl'}

In [3]:
import json
from pathlib import Path

first = Path(paths["train"]).read_text(encoding="utf-8").splitlines()[0]
print(json.dumps(json.loads(first), indent=2, ensure_ascii=False))

{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "If 5 notebooks cost $32.50, how much does one notebook cost in dollars?"
        }
      ]
    }
  ],
  "references": {
    "ground_truth_answer": "6.50"
  },
  "systemInstruction": {
    "parts": [
      {
        "text": "You are a careful math tutor. Think step by step, then end with the final answer on its own line as 'Answer: <number>'."
      }
    ]
  }
}


## 2. The reward function

The reward is a real, unit-tested Python function in `geap_tuning.rlft.reward`.
It is shipped **verbatim** to the GEAP code-execution sandbox as
`python_code_snippet` (so it must be stdlib-only) **and** reused for offline
eval — one tested function, two uses. The sandbox calls
`evaluate(example, response) -> float`, clipped to `[-1, 1]`.

In [4]:
from geap_tuning.rlft import reward

# Local sanity check: a correct answer earns +1, a wrong one -1.
example = {"references": {"ground_truth_answer": "391"}}
good = {"parts": [{"text": "17 * 23 = 391\nAnswer: 391"}]}
bad = {"parts": [{"text": "Answer: 40"}]}
reward.evaluate(example, good), reward.evaluate(example, bad)

(1.0, -1.0)

## 3. Stage the splits to GCS

In [5]:
from geap_tuning.gcs import upload_file

train_uri = upload_file(paths["train"], f"{cfg.bucket}/rlft_math/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/rlft_math/val.jsonl")
train_uri, val_uri

('gs://geap-tuning-hybrid-vertex/rlft_math/train.jsonl',
 'gs://geap-tuning-hybrid-vertex/rlft_math/val.jsonl')

## 4. Preflight the reward

Before spending money, validate the reward on one example via
`tunings.validate_reward`. RLFT auto-stops if >80% of reward calls error, so a
broken reward is worth catching here.

In [6]:
from geap_tuning.rlft.data import MATH_PROBLEMS, build_rlft_records, split_dataset
from geap_tuning.rlft.tune import validate_reward_config

train_records = build_rlft_records(split_dataset(MATH_PROBLEMS)[0])

preflight = validate_reward_config(
    client,
    project=cfg.project,
    location=cfg.location,  # regional — validate_reward is not offered on `global`
    sample_answer="Answer: 4",
    example_record=train_records[0],
)

preflight

ValidateRewardResponse(
  overall_reward=-1.0,
  reward_info_details={
    'math_correctness': ReinforcementTuningRewardInfo(
      reward=-1.0
    )
  },
  sdk_http_response=HttpResponse(
    headers=<dict len=9>
  )
)

## 5. Launch the RLFT job and wait

Reuse an existing job with the same display name if one exists (cost control).
`samples_per_prompt` is how many candidate generations are drawn per prompt for
reward comparison.

> This demo tunes `gemini-3.5-flash` (`BASE_MODEL`, set in the config cell) in a
> **regional** location. Gemini 3.x is served from the `global` endpoint for
> *inference*, but the `global` endpoint **does not support tuning** — so the
> tuning job and the `validate_reward` preflight above both run in `cfg.location`
> (`us-central1`; `europe-west4` also works).

In [7]:
VERSION = "v1"

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, wait_for_tuning_job
from geap_tuning.rlft.tune import launch_rlft_job

DISPLAY_NAME = f"geap-rlft-math-{VERSION}"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_rlft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        labels=cfg.labels,
    )
job = wait_for_tuning_job(client, job.name)
job.state

Type mismatch in CreateTuningJobConfig.validation_dataset: expected TuningValidationDataset, got TuningDataset
/home/user/geap_tuning/src/geap_tuning/rlft/tune.py:77: ExperimentalWarning: The SDK's tuning implementation is experimental, and may change in future versions.
  return client.tunings.tune(


## 6. Evaluate the tuned endpoint

RLFT has no single gold answer, but `references` carries ground truth — so we
reuse the **same reward** for offline eval: generate a reply per held-out
prompt and report the fraction that earns a positive reward (answer accuracy).

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.jobs import tuned_endpoint
from geap_tuning.rlft.evaluate import run_rlft_eval

endpoint = tuned_endpoint(job)
_, _, test_problems = split_dataset(MATH_PROBLEMS)
metrics = run_rlft_eval(
    build_rlft_records(test_problems),
    generate_fn=lambda user_text: generate(client, endpoint, user_text),
)
print(f"Held-out answer accuracy: {metrics['accuracy']:.3f} (n={metrics['n']})")

## Next steps

All three GEAP tuning services are now demonstrated end to end — **SFT**
([01](01_sft.ipynb)), **DPO** ([02](02_preference_tuning.ipynb)), and **RLFT**
(this notebook). From here you can combine them: SFT to teach a skill, then
continuous-tune with DPO or RLFT to refine style or correctness.